In [1]:

import numpy as np
from collections import deque

class AdjustPidController:

    _errors = deque(maxlen=100)
    _rate = 0.01

    def _reset(self):
        self._errors.clear()
        self._rate = 0.01
    
    def _adjust1(self, error, params):

        self._errors.append(error)

        if len(self._errors) < 50:            
            return None

        recent_errors = list(self._errors)[-20:]
        previous_errors = list(self._errors)[-40:-20]

        if len(previous_errors) == 0:
            return None
        
        Kp = params[0]

        recent_mse = np.mean(np.square(recent_errors))
        previous_mse = np.mean(np.square(previous_errors))
    
        improvement = (previous_mse - recent_mse) / (previous_mse + 1e-6)

        if improvement > 0.1:
            self._rate *= 1.1
        elif improvement < -0.1:
            self._rate *= 0.9

        self._rate = np.clip(self._rate, 0.001, 0.1)

        if recent_mse > previous_mse * 1.1:
            direction = -np.sign(self._change)
        else:
            direction = np.sign(self._change) if hasattr(self, '_change') else 1
                
        kp_ *= (1 + direction * self._rate)
        kp_ = np.clip(kp_, 200.0, 2000.0)

        self._change = kp_ - Kp

        print(f"{self._rate} {kp_}")

        return kp_
    
    def _adjust2(self, setpoint, error, params):

        self._errors.append(error)

        if len(self._errors) < 50:
            return None
        
        rate = 1.01

        recent_errors = list(self._errors)[-20:]

        error_trend = np.polyfit(range(len(recent_errors)), recent_errors, 1)[0]
        error_mean = np.mean(np.abs(recent_errors))
        error_amplitude = np.std(recent_errors)

        kp = params[0]

        # oszilate
        zero_crossings = sum(1 for i in range(1, len(recent_errors)) if recent_errors[i-1] * recent_errors[i] < 0)
        
        if zero_crossings > 5 and error_amplitude > 0.1 * abs(setpoint):
            kp /= rate
                            
        # to slow
        elif abs(error_trend) < 0.01 and error_mean > 0.2 * abs(setpoint):
            kp *= rate
                            
        # error decreases - increase slighly
        elif abs(error) < 0.5 * error_mean and error_mean > 0:
            kp *= rate
            
        # error to high
        elif abs(error) > 0.5 * abs(setpoint):
            kp *= rate

        else:
            kp /= rate

        return kp